In [1]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [3]:
from fastapi import APIRouter,HTTPException,Request
from agents.supervisor import getAgent
from agents.state import AgentState
from services.redisclient import cacheGet,cacheSet
from config.settings import settings
from config.constants import INTENT_STOCK_PREDICT
from models.schemas import validateStockPredictRequest
import time,logging

In [4]:
router=APIRouter()
logger=logging.getLogger(__name__)

In [7]:
@router.post("/stockpredict/{stockId}")
async def predictStock(stockId:str,request:Request):
    body=await request.json()
    params=validateStockPredictRequest(body)
    cacheKey=f"stock-predict-{stockId}"
    cached=await cacheGet(cacheKey)
    if cached:
        return{
            "success":True,
            "data":cached,
            "error":""
        }
    agent=getAgent()
    if agent is None:
        raise HTTPException(status_code=500,detail="Agent not available")
    startTime=time.time()
    try:
        state=AgentState(
            query=params["query"] or f"Predict stock {stockId}",
            intent=INTENT_STOCK_PREDICT,
            confidence=1.0,
            slots={
                "stockId":stockId
            },
            context={
                **params["context"],
                "stockId":stockId,
                "pageType":"stock"
            },
            user_id=params["userId"]
        )
        result=await agent.ainvoke(state)
    except Exception as e:
        logger.error(f"stock prediction error {e}")
        raise HTTPException(status_code=500,detail=str(e))
    output=result.get("output",{})
    output["_meta"]={
        **(output.get("_meta",{})),
        "latencyMs":round((time.time()-startTime)*1000,2),
        "stockId":stockId
    }
    await cacheSet(
        cacheKey,
        output,
        ttl=settings.STOCK_CACHE_TTL
    )
    return{
        "success":True,
        "data":output,
        "error":""
    }